# AIC25 — Multi-Camera Tracking Pipeline
**Subproject 1: Single-Camera Tracking Consistency**

Runs on both Google Colab (GPU) and locally in VS Code (CPU).  
Execute cells in order from top to bottom.

---
## Step 0 — Detect environment

In [ ]:
import os, sys

ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_COLAB:
    REPO   = '/content/repo'
    PYTHON = 'python'
    print('Environment : Google Colab')
else:
    REPO   = '/home/seco/deepLearning/Single-Camera-Tracking-Consistency'
    PYTHON = f'{REPO}/.venv/bin/python'
    os.chdir(REPO)
    print('Environment : Local VS Code')

print(f'REPO        : {REPO}')
print(f'Python      : {PYTHON}')

---
## Step 1 — Clone repo and install dependencies
*(Colab only — skipped locally)*

In [ ]:
if ON_COLAB:
    # Clone
    if not os.path.exists(REPO):
        os.system(f'git clone https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git {REPO}')
        print('Repo cloned.')
    else:
        os.system(f'git -C {REPO} pull')
        print('Repo updated.')

    os.chdir(REPO)

    # BoT-SORT
    print('Installing BoT-SORT dependencies...')
    os.system(f'pip install -q -r {REPO}/BoT-SORT/requirements.txt')
    os.system('pip install -q cython_bbox pycocotools faiss-gpu cmake')
    os.chdir(f'{REPO}/BoT-SORT')
    os.system('python setup.py develop --quiet')

    # torchreid
    print('Installing torchreid...')
    os.system(f'pip install -q -r {REPO}/deep-person-reid/requirements.txt')
    os.chdir(f'{REPO}/deep-person-reid')
    os.system('python setup.py develop --quiet')

    # tracking
    os.chdir(REPO)
    os.system(f'pip install -q -r {REPO}/tracking/requirements.txt')

    print('All dependencies installed.')
else:
    print('Local: using existing .venv — skipping install.')

---
## Step 2 — Check GPU

In [ ]:
import subprocess

result = subprocess.run(
    [PYTHON, '-c',
     'import torch; '
     'print("CUDA available:", torch.cuda.is_available()); '
     'print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (CPU only)")'],
    capture_output=True, text=True
)
print(result.stdout.strip())
if not torch_ok := 'CUDA: True' not in result.stdout and ON_COLAB:
    print('WARNING: No GPU detected on Colab. Go to Runtime > Change runtime type > T4 GPU')

---
## Step 3 — Download pretrained models
*(Colab only — skipped locally)*

In [ ]:
if ON_COLAB:
    os.system('pip install -q gdown')

    # OSNet Re-ID model
    osnet_path = f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar'
    os.makedirs(os.path.dirname(osnet_path), exist_ok=True)
    if not os.path.exists(osnet_path):
        print('Downloading OSNet...')
        os.system(f'gdown "https://drive.google.com/uc?id=1IosIFlLiulGIjwW3H8uMCC3YvMyr9gZ2" -O {osnet_path}')
    else:
        print('OSNet already downloaded.')

    # ByteTrack base detector (YOLOX_x trained on MOT17 — persons only)
    bytetrack_path = f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'
    os.makedirs(os.path.dirname(bytetrack_path), exist_ok=True)
    if not os.path.exists(bytetrack_path):
        print('Downloading ByteTrack base model...')
        os.system(f'gdown "https://drive.google.com/uc?id=1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5" -O {bytetrack_path}')
    else:
        print('ByteTrack model already downloaded.')

    # AIC25 trained detector (if already trained and saved to Drive)
    from google.colab import drive
    drive.mount('/content/drive')
    ai_src = '/content/drive/MyDrive/AIC25/models/ai_city_ckpt.pth.tar'
    ai_dst = f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'
    if os.path.exists(ai_src):
        os.system(f'cp {ai_src} {ai_dst}')
        print('AIC25 detector copied from Drive.')
    else:
        print('AIC25 detector not found on Drive — using base model (persons only).')
else:
    print('Local: models already in place.')

---
## Step 4 — Configure scene
Change `SCENE` here to run a different warehouse.

In [ ]:
SCENE   = 'Warehouse_016'
DATASET = 'Val'            # 'Val' or 'Train'

os.chdir(REPO)
data_dir = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
print(f'Scene   : {SCENE}')
print(f'Dataset : {DATASET}')
print(f'Data dir exists: {os.path.exists(data_dir)}')
if os.path.exists(data_dir):
    print(f'Contents: {os.listdir(data_dir)}')

---
## Step 5 — Download dataset from HuggingFace
*(Colab only — skipped locally)*

> Full dataset is 3.31 TB. Only videos + calibration + ground_truth are downloaded (~7 GB per scene).  
> Depth maps are skipped to save Colab disk space.

In [ ]:
if ON_COLAB:
    os.system('pip install -q huggingface_hub')
    from huggingface_hub import login, snapshot_download
    import shutil

    login()  # enter your HuggingFace token when prompted

    hf_split = DATASET.lower()
    print(f'Downloading {SCENE} ({DATASET})...')
    snapshot_download(
        repo_id='nvidia/PhysicalAI-SmartSpaces',
        repo_type='dataset',
        local_dir='/content/hf_data',
        allow_patterns=[
            f'MTMC_Tracking_2025/{hf_split}/{SCENE}/videos/**',
            f'MTMC_Tracking_2025/{hf_split}/{SCENE}/calibration.json',
            f'MTMC_Tracking_2025/{hf_split}/{SCENE}/ground_truth.json',
        ]
    )

    # Remap HuggingFace paths to what the code expects
    # HuggingFace: val/ (lowercase)  →  code expects: Val/ (capital)
    HF  = '/content/hf_data/MTMC_Tracking_2025'
    src = f'{HF}/{hf_split}/{SCENE}'
    dst = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
    os.makedirs(dst, exist_ok=True)

    for fname in ['calibration.json', 'ground_truth.json']:
        if os.path.exists(f'{src}/{fname}'):
            shutil.copy(f'{src}/{fname}', f'{dst}/{fname}')

    if not os.path.exists(f'{dst}/videos'):
        os.symlink(f'{src}/videos', f'{dst}/videos')

    os.makedirs(f'{dst}/depth_map', exist_ok=True)
    print('Dataset ready.')
else:
    print('Local: using existing AIC25_Track1/ data.')

---
## [A] Train detector — skip if `ai_city_ckpt.pth.tar` already exists
Only needed once. Takes ~6–10 hours on T4 GPU.  
**Skip this section if the model is already on Drive.**

In [ ]:
# Save training outputs to Drive so they survive session disconnects
if ON_COLAB:
    os.makedirs('/content/drive/MyDrive/AIC25/YOLOX_outputs', exist_ok=True)
    os.system(f'ln -sfn /content/drive/MyDrive/AIC25/YOLOX_outputs {REPO}/YOLOX_outputs')

os.chdir(REPO)

# Extract training frames
os.system(f'{PYTHON} tools/extract_frames_25.py ./AIC25_Track1/Train -s {SCENE}')

# Generate COCO annotations
os.system(f'{PYTHON} tools/convert_to_coco_25.py --BASE_DIR ./AIC25_Track1 -s Train')

# Train — fine-tunes from bytetrack_x_mot17.pth.tar
os.system(f'{PYTHON} BoT-SORT/yolox/train.py '
          f'-f BoT-SORT/yolox/exps/example/mot/yolox_x_AI_City_25.py '
          f'-d 1 -b 4 --fp16 '
          f'-c BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar')

# Save best checkpoint to Drive
if ON_COLAB:
    os.makedirs('/content/drive/MyDrive/AIC25/models', exist_ok=True)
    os.system(f'cp {REPO}/YOLOX_outputs/yolox_x_AI_City_25/best_ckpt.pth.tar {REPO}/BoT-SORT/ai_city_ckpt.pth.tar')
    os.system(f'cp {REPO}/YOLOX_outputs/yolox_x_AI_City_25/best_ckpt.pth.tar /content/drive/MyDrive/AIC25/models/ai_city_ckpt.pth.tar')
    print('Training complete — checkpoint saved to Drive.')

---
## [B] Extract frames from video

In [ ]:
os.chdir(REPO)
ret = os.system(f'{PYTHON} tools/extract_frames_25.py ./AIC25_Track1/{DATASET} -s {SCENE}')
print('Done' if ret == 0 else f'ERROR code {ret}')

---
## [C] Detection — YOLOX
Detects objects in every frame of every camera.  
To test on 1 camera / 100 frames first, uncomment the quick-test command.

In [ ]:
os.chdir(f'{REPO}/BoT-SORT')

# Full run — all cameras, all frames
cmd = f'{PYTHON} tools/aic25_get_detection.py --scene {SCENE} --dataset {DATASET} ../'

# Quick test — 1 camera, 100 frames (uncomment to use)
# cmd = f'{PYTHON} tools/aic25_get_detection.py --scene {SCENE} --dataset {DATASET} --camera Camera --max_frames 100 ../'

print(f'Running: {cmd}')
ret = os.system(cmd)
print('Done' if ret == 0 else f'ERROR code {ret}')

---
## [D] Re-ID embeddings — OSNet
Extracts appearance features for each detected object crop.

In [ ]:
os.chdir(f'{REPO}/deep-person-reid')
ret = os.system(f'{PYTHON} torchreid/aic25_extract.py -s {SCENE} ../')
print('Done' if ret == 0 else f'ERROR code {ret}')

---
## [E] Single-camera tracking — all cameras
Runs BoT-SORT on each camera independently.

In [ ]:
os.chdir(REPO)
videos_dir = f'AIC25_Track1/{DATASET}/{SCENE}/videos'
cameras = sorted([d for d in os.listdir(videos_dir) if os.path.isdir(f'{videos_dir}/{d}')])
print(f'Found {len(cameras)} cameras: {cameras}\n')

for cam in cameras:
    print(f'--- Tracking {cam} ---')
    ret = os.system(f'{PYTHON} BoT-SORT/single_camera_tracking.py -s {SCENE} -c {cam} --dataset {DATASET}')
    print('OK' if ret == 0 else f'ERROR {ret}')

---
## [F] Fix single-camera results
Cleans up tracklets: removes short tracks, fills gaps, applies Sequential NMS.

In [ ]:
os.chdir(REPO)
videos_dir = f'AIC25_Track1/{DATASET}/{SCENE}/videos'
cameras = sorted([d for d in os.listdir(videos_dir) if os.path.isdir(f'{videos_dir}/{d}')])

for cam in cameras:
    print(f'--- Fixing {cam} ---')
    ret = os.system(f'{PYTHON} BoT-SORT/single_camera_fix.py -s {SCENE} -c {cam} --dataset {DATASET}')
    print('OK' if ret == 0 else f'ERROR {ret}')

---
## [G] Multi-camera tracking
Assigns global IDs across all cameras using Glance initialization + progressive association.

In [ ]:
os.chdir(REPO)
ret = os.system(f'{PYTHON} BoT-SORT/multi_camera_revised.py -s {SCENE} --dataset {DATASET}')
print('Done' if ret == 0 else f'ERROR code {ret}')

In [ ]:
os.chdir(REPO)
ret = os.system(f'{PYTHON} BoT-SORT/multi_camera_fix.py -s {SCENE} --dataset {DATASET}')
print('Done' if ret == 0 else f'ERROR code {ret}')

---
## [H] Evaluate — HOTA3D
Converts outputs to TrackEval format and computes HOTA3D metrics.

In [ ]:
os.chdir(f'{REPO}/TrackEval')
ret = os.system(f'{PYTHON} prepare_eval_data.py -s {SCENE} --exp exp1 --base_dir {REPO}')
print('GT prepared' if ret == 0 else f'ERROR code {ret}')

In [ ]:
os.chdir(f'{REPO}/TrackEval')
ret = os.system(f'{PYTHON} main.py -s {SCENE} --exp exp1')
print('Evaluation complete' if ret == 0 else f'ERROR code {ret}')

---
## Check results

In [ ]:
import json

os.chdir(REPO)
results = {}
videos_dir = f'AIC25_Track1/{DATASET}/{SCENE}/videos'
cameras = sorted([d for d in os.listdir(videos_dir) if os.path.isdir(f'{videos_dir}/{d}')])

for cam in cameras:
    fixed = f'Tracking/Singlecamera/{SCENE}/{cam}/fixed_{cam}.json'
    if os.path.exists(fixed):
        with open(fixed) as f:
            data = json.load(f)
        ids = set()
        for tracks in data.values():
            for t in tracks:
                ids.add(t.get('object sc id'))
        results[cam] = {'frames': len(data), 'tracklets': len(ids)}
    else:
        results[cam] = 'not yet run'

print(f'{'Camera':<12} {'Frames':>8} {'Tracklets':>10}')
print('-' * 32)
for cam, r in results.items():
    if isinstance(r, dict):
        print(f'{cam:<12} {r["frames"]:>8} {r["tracklets"]:>10}')
    else:
        print(f'{cam:<12}   {r}')